# Bài 7: Khi AI thiên vị
**Mở hộp đen AI · Tập 1 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**. Tin nhắn trong bài do nhóm tác giả tự soạn bằng cách ghép cụm từ, giống Bài 5.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 1. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Các cụm từ để ghép tin nhắn (giống Bài 5) và hàm bỏ dấu

In [ ]:
MO_RAC = ["chúc mừng bạn", "thông báo khẩn", "ưu đãi hôm nay",
          "tài khoản của bạn", "bạn có bưu phẩm", "mẹ ơi con đây",
          "em chào anh chị", "shop báo", "ngân hàng thông báo", "bạn ơi",
          "anh ơi"]
THAN_RAC = ["trúng thưởng xe máy", "được vay tiền lãi thấp",
            "bị khóa cần xác minh", "nhận quà miễn phí",
            "việc nhẹ lương cao tại nhà", "nạp thẻ nhận gấp đôi",
            "con đổi số mới chuyển tiền giúp con",
            "đơn hàng chưa giao cần xác nhận", "nhận tiền hoàn chưa",
            "mã xác nhận sắp hết hạn"]
KET_RAC = ["bấm link ngay", "liên hệ ngay", "trả lời tin này",
           "bấm vào link để nhận", "gọi số này nhé", "nhé", "ạ",
           "bấm link để xem"]

MO_THUONG = ["mẹ ơi", "con ơi", "bạn ơi", "cô thông báo", "lớp mình", "shop báo",
             "ngân hàng thông báo", "anh ơi", "thông báo"]
THAN_THUONG = ["mai kiểm tra toán", "tối nay học nhóm", "nhớ mang áo mưa",
               "đơn hàng đã giao", "chiều đá bóng", "con về muộn",
               "nhận tiền chưa", "mã xác nhận đã gửi",
               "họp phụ huynh sáng chủ nhật", "gửi mình bài tập với",
               "sinh nhật lan tối thứ bảy", "chuyển tiền học thêm rồi",
               "có lịch thi mới", "nhận quà sinh nhật chưa",
               "trúng thưởng bốc thăm của lớp rồi",
               "nhận quà miễn phí ở căng tin", "tài khoản học trực tuyến đã mở",
               "vay tiền mua sách được không"]
KET_THUONG = ["nhé", "nha", "không", "nhé con", "ạ", "bấm link để xem",
              "liên hệ cô nhé", "trả lời mẹ nhé", "liên hệ ngay",
              "trả lời tin này"]

import random, unicodedata

def bo_dau(chu):
    # "Mẹ ơi" -> "Me oi": tách dấu khỏi chữ cái rồi bỏ dấu đi
    chu = chu.replace("đ", "d").replace("Đ", "D")
    tach = unicodedata.normalize("NFD", chu)
    return "".join(k for k in tach if unicodedata.category(k) != "Mn")

def ghep(mo, than, ket):
    return f"{random.choice(mo)} {random.choice(than)} {random.choice(ket)}"

print(bo_dau("mẹ ơi tối nay học nhóm nhé"))

### Ô 2. Dữ liệu huấn luyện bị lệch
Hầu hết tin thường mà nhóm thu được là của các bạn gõ **có dấu**. Chỉ 10 tin thường gõ **không dấu**, như cách ông bà hay điện thoại đời cũ vẫn nhắn. Còn tin rác thì phần lớn **không dấu**, vì kẻ gửi rác hay viết như vậy để lách bộ lọc.

In [ ]:
random.seed(7)
tin_rac = set(); tin_thuong = set()
while len(tin_rac) < 50:
    tin_rac.add(ghep(MO_RAC, THAN_RAC, KET_RAC))
while len(tin_thuong) < 280:
    tin_thuong.add(ghep(MO_THUONG, THAN_THUONG, KET_THUONG))
tin_rac, tin_thuong = sorted(tin_rac), sorted(tin_thuong)
random.shuffle(tin_rac); random.shuffle(tin_thuong)

# Tập huấn luyện: lệch về nhóm có dấu ở tin thường
hoc = ([(t, "thường") for t in tin_thuong[:150]] +             # có dấu
       [(bo_dau(t), "thường") for t in tin_thuong[150:160]] +   # không dấu: 10 tin
       [(t, "rác") for t in tin_rac[:12]] +
       [(bo_dau(t), "rác") for t in tin_rac[12:30]])

# Bộ kiểm tra công bằng: mỗi nhóm có số tin như nhau, toàn tin máy chưa thấy
thi = ([(t, "thường", "có dấu") for t in tin_thuong[160:190]] +
       [(bo_dau(t), "thường", "không dấu") for t in tin_thuong[190:220]] +
       [(t, "rác", "có dấu") for t in tin_rac[30:40]] +
       [(bo_dau(t), "rác", "không dấu") for t in tin_rac[40:50]])
print("Học:", len(hoc), "tin | Kiểm tra:", len(thi), "tin")

### Ô 3. Huấn luyện và chấm điểm chung

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

def huan_luyen(du_lieu):
    dem = CountVectorizer()
    X = dem.fit_transform([t for t, n in du_lieu])
    mo_hinh = MultinomialNB().fit(X, [n for t, n in du_lieu])
    return dem, mo_hinh

dem, mo_hinh = huan_luyen(hoc)
doan = mo_hinh.predict(dem.transform([t for t, n, g in thi]))
dung = sum(d == n for d, (t, n, g) in zip(doan, thi))
print(f"Độ chính xác chung: {dung / len(thi):.1%}")

### Ô 4. Mổ xẻ theo từng nhóm

In [ ]:
def bang_cong_bang(dem, mo_hinh, bien_doi=lambda t: t):
    doan = mo_hinh.predict(dem.transform([bien_doi(t) for t, n, g in thi]))
    print("nhóm       | chặn nhầm tin thường | bắt được tin rác")
    for nhom in ["có dấu", "không dấu"]:
        thuong = [d for d, (t, n, g) in zip(doan, thi)
                  if g == nhom and n == "thường"]
        rac = [d for d, (t, n, g) in zip(doan, thi)
               if g == nhom and n == "rác"]
        nham, bat = thuong.count("rác"), rac.count("rác")
        print(f"{nhom:10} | {nham:>2} / {len(thuong)} = {nham / len(thuong):5.1%}"
              f"      | {bat:>2} / {len(rac)} = {bat / len(rac):5.1%}")

bang_cong_bang(dem, mo_hinh)

## Chặng 2: Đọc
Trả lời vào vở trước khi đi tiếp:
1. Độ chính xác chung ở ô 3 có cho em thấy vấn đề ở ô 4 không?
2. Nhóm nào bị chặn nhầm nhiều hơn? Họ có làm gì sai không?
3. Dữ liệu không hề có cột 'nhóm người dùng'. Vậy máy phân biệt hai nhóm bằng cách nào?

## Chặng 3: Sửa
### Ô 5. Máy đã học những 'từ' nào là dấu hiệu tin rác?

In [ ]:
import numpy as np
tu = dem.get_feature_names_out()
nghieng = mo_hinh.feature_log_prob_[0] - mo_hinh.feature_log_prob_[1]   # 0 là 'rác'
print("10 từ nghiêng về RÁC nhất:", list(tu[np.argsort(nghieng)[-10:][::-1]]))

### Ô 6. Cách sửa 1: thu thêm dữ liệu của nhóm bị thiếu
Thêm 60 tin thường gõ không dấu vào tập huấn luyện (vẫn không trùng bộ kiểm tra).

In [ ]:
hoc_them = hoc + [(bo_dau(t), "thường") for t in tin_thuong[220:280]]
dem2, mo_hinh2 = huan_luyen(hoc_them)
bang_cong_bang(dem2, mo_hinh2)

### Ô 7. Cách sửa 2: bỏ dấu hết trước khi học
Nếu mọi tin đều không dấu thì máy không còn dùng dấu để phân biệt được nữa.

In [ ]:
hoc_bo_dau = [(bo_dau(t), n) for t, n in hoc]
dem3, mo_hinh3 = huan_luyen(hoc_bo_dau)
bang_cong_bang(dem3, mo_hinh3, bien_doi=bo_dau)

## Chặng 4: Tự làm
1. Ở ô 6, thay 60 bằng 20 rồi 40 tin. Tỉ lệ chặn nhầm của nhóm không dấu giảm thế nào?
2. Ở đây cách sửa 2 còn làm tốt hơn cả. Vậy có nên lúc nào cũng xoá bớt thông tin trước khi học không? Gợi ý: trong tiếng Việt, 'ma', 'má', 'mà' khác nghĩa nhau.
3. Nghĩ ra một nhóm người dùng khác có thể bị bộ lọc đối xử thiệt thòi (ví dụ người hay viết tắt, người dùng tiếng địa phương). Em sẽ kiểm tra bằng cách nào?

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*